# Research Module 3: Machine Learning & Image Processing

**AI-Ready Radiology Curriculum**

In this notebook you will:
1. Build machine learning classifiers on radiology data (logistic regression, decision tree)
2. Load, display, and manipulate images in Python
3. Run a pre-trained neural network (MobileNetV2) to classify photos
4. Use MediaPipe to detect hand landmarks, segment hands, and measure finger lengths

---

**Prerequisites:** Research Modules 1 and 2 completed (Python basics, pandas, evaluation metrics).

---

## Setup

Install MediaPipe (everything else is pre-installed on Colab).

In [ ]:
!pip install -q mediapipe

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from PIL import Image, ImageEnhance, ImageFilter
import requests
from io import BytesIO
from IPython.display import display


def load_image_from_url(url):
    """Load a PIL Image from a URL (sets User-Agent to avoid 403 errors)."""
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert('RGB')


print('All libraries loaded successfully.')

In [ ]:
# ============================================================
# IMPORTANT: Replace YOUR-USERNAME with your GitHub username
# ============================================================
GITHUB_USERNAME = 'YOUR-USERNAME'

BASE_URL = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/Bootcamp-AI-for-Medical-Imaging/main/data'
print(f'Base URL: {BASE_URL}')

---

## Part 1: Introduction to Machine Learning

Machine learning (ML) is a method for building systems that learn patterns from data rather than being explicitly programmed.

There are two main types:
- **Supervised learning** — You provide labeled examples (input + correct answer). The model learns to predict the answer for new inputs. Example: given AI confidence scores and modality, predict whether a radiologist will confirm the finding.
- **Unsupervised learning** — No labels. The model finds structure on its own. Example: grouping similar imaging studies together.

In this module we focus on **supervised classification**: predicting a category (confirmed vs. not confirmed) from input features.

### 1.1 Load and Prepare the Data

We will use the same radiology AI findings dataset from Module 2. ML models need numeric inputs, so we convert categorical columns (like modality) into numbers using **one-hot encoding**.

In [ ]:
df = pd.read_csv(f'{BASE_URL}/radiology_ai_findings.csv')
print(f'Loaded {len(df)} studies')
df.head()

In [ ]:
# Select features and target
y = df['radiologist_confirmed'].astype(int)

features = df[['ai_confidence', 'modality']].copy()
features_encoded = pd.get_dummies(features, columns=['modality'], dtype=int)

print(f'Features shape: {features_encoded.shape}')
print(f'Feature columns: {list(features_encoded.columns)}')
features_encoded.head()

### 1.2 Train/Test Split

You never evaluate a model on the same data it learned from — that would be like giving a student the exam answers during study, then testing them with the same questions.

We split the data:
- **Training set** (80%) — the model learns from this
- **Test set** (20%) — held out to measure real performance

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features_encoded, y, test_size=0.2, random_state=42
)

print(f'Training set: {len(X_train)} samples')
print(f'Test set:     {len(X_test)} samples')
print(f'Features per sample: {X_train.shape[1]}')

### 1.3 Logistic Regression

Logistic regression is one of the simplest classifiers. Despite its name, it is used for **classification** (not regression). It learns a weighted combination of features and outputs a probability between 0 and 1.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, lr_predictions)
print(f'Logistic Regression Accuracy: {lr_accuracy:.1%}')
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, lr_predictions))
print()
print('Classification Report:')
print(classification_report(y_test, lr_predictions, target_names=['Not Confirmed', 'Confirmed']))

### 1.4 Decision Tree

A decision tree learns a series of yes/no questions about the features. It splits the data at each step to separate the classes as cleanly as possible.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt_model = DecisionTreeClassifier(random_state=42, max_depth=3)
dt_model.fit(X_train, y_train)

dt_predictions = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_predictions)
print(f'Decision Tree Accuracy: {dt_accuracy:.1%}')
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, dt_predictions))
print()
print('Classification Report:')
print(classification_report(y_test, dt_predictions, target_names=['Not Confirmed', 'Confirmed']))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(dt_model, feature_names=list(features_encoded.columns),
          class_names=['Not Confirmed', 'Confirmed'],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree: Predicting Radiologist Confirmation', fontsize=14)
plt.tight_layout()
plt.show()

### 1.5 Compare Models

In [ ]:
import seaborn as sns

print('=== Model Comparison ===')
print(f'Logistic Regression: {lr_accuracy:.1%}')
print(f'Decision Tree:       {dt_accuracy:.1%}')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name, preds in zip(axes,
                           ['Logistic Regression', 'Decision Tree'],
                           [lr_predictions, dt_predictions]):
    cm = confusion_matrix(y_test, preds)
    labels = np.array([[f'TN\n{cm[0,0]}', f'FP\n{cm[0,1]}'],
                       [f'FN\n{cm[1,0]}', f'TP\n{cm[1,1]}']])
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', cbar=False,
                xticklabels=['Not Confirmed', 'Confirmed'],
                yticklabels=['Predicted: No', 'Predicted: Yes'],
                annot_kws={'size': 13, 'fontweight': 'bold'}, ax=ax)
    acc = accuracy_score(y_test, preds)
    ax.set_title(f'{name}\nAccuracy: {acc:.1%}', fontsize=12)

plt.tight_layout()
plt.show()

---

## Part 2: Image Basics in Python

AI in radiology works with images. Before using pre-trained models, you need to understand how computers represent images: as grids of numbers (pixel arrays).

### 2.1 Load and Display an Image

We load a chest X-ray from your GitHub repository using PIL (Python Imaging Library) and matplotlib.

In [ ]:
img_url = f'{BASE_URL}/CXR.jpg'
sample_img = load_image_from_url(img_url)

print(f'Image size: {sample_img.size} (width x height)')
print(f'Image mode: {sample_img.mode}')

plt.figure(figsize=(6, 6))
plt.imshow(sample_img, cmap='gray')
plt.title('Sample Chest X-Ray')
plt.axis('off')
plt.show()

### 2.2 Understanding Pixel Arrays

Every image is stored as a NumPy array of numbers. Each number represents the brightness of one pixel.

- **Grayscale images** have shape `(height, width)` — one number per pixel (0 = black, 255 = white)
- **Color images** have shape `(height, width, 3)` — three numbers per pixel (Red, Green, Blue)

In [ ]:
img_array = np.array(sample_img)

print(f'Array shape: {img_array.shape}')
print(f'Data type:   {img_array.dtype}')
print(f'Min value:   {img_array.min()}')
print(f'Max value:   {img_array.max()}')
print()
print('Top-left 5x5 pixel values:')
print(img_array[:5, :5] if img_array.ndim == 2 else img_array[:5, :5, 0])

### 2.3 Basic Image Operations

Before feeding images to a model, you typically resize them, adjust contrast, or convert to grayscale.

In [ ]:
resized = sample_img.resize((224, 224))
grayscale = sample_img.convert('L')
bright = ImageEnhance.Brightness(sample_img).enhance(1.5)
high_contrast = ImageEnhance.Contrast(sample_img).enhance(2.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, title, img in zip(axes,
                          ['Resized (224x224)', 'Grayscale', 'Brighter (1.5x)', 'High Contrast (2x)'],
                          [resized, grayscale, bright, high_contrast]):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f'Original size:  {sample_img.size}')
print(f'Resized:        {resized.size}')
print(f'Grayscale mode: {grayscale.mode}')

---

## Part 3: Pre-Trained Image Classification

Training a neural network from scratch requires millions of images and days of computing time. **Transfer learning** lets you use a model someone else already trained and apply it to your own images.

We will use **MobileNetV2**, a lightweight neural network trained on **ImageNet** (a dataset of 1.2 million images across 1,000 everyday categories). MobileNetV2 was developed by Google and is widely used in mobile and embedded applications.

This is the same basic approach used in radiology AI — except medical models are fine-tuned on X-rays and CT scans instead of everyday photos.

### 3.1 Load MobileNetV2

In [ ]:
import torch
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

weights = MobileNet_V2_Weights.IMAGENET1K_V1
model = mobilenet_v2(weights=weights)
model.eval()

preprocess = weights.transforms()
categories = weights.meta['categories']

print(f'Model loaded: MobileNetV2')
print(f'Number of output classes: {len(categories)}')
print(f'First 10 classes: {categories[:10]}')

### 3.2 Classify a Sample Image

Let's download a photo and see what MobileNetV2 thinks it is.

In [ ]:
sample_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/500px-YellowLabradorLooking_new.jpg'
photo = load_image_from_url(sample_url)

plt.figure(figsize=(5, 5))
plt.imshow(photo)
plt.title('Sample Photo')
plt.axis('off')
plt.show()

In [ ]:
input_tensor = preprocess(photo).unsqueeze(0)

with torch.no_grad():
    output = model(input_tensor)

probabilities = torch.nn.functional.softmax(output[0], dim=0)
top5_prob, top5_idx = torch.topk(probabilities, 5)

print('=== Top 5 Predictions ===')
for i in range(5):
    class_name = categories[top5_idx[i].item()]
    confidence = top5_prob[i].item()
    print(f'{i+1}. {class_name:30s} {confidence:.1%}')

In [ ]:
names = [categories[idx.item()] for idx in top5_idx]
probs = [p.item() for p in top5_prob]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#0EA5E9' if i == 0 else '#CBD5E1' for i in range(5)]
ax.barh(range(4, -1, -1), probs, color=colors)
ax.set_yticks(range(4, -1, -1))
ax.set_yticklabels(names)
ax.set_xlabel('Confidence')
ax.set_title('MobileNetV2 Top-5 Predictions')
ax.set_xlim(0, 1)

for i, (p, name) in enumerate(zip(probs, names)):
    ax.text(p + 0.01, 4 - i, f'{p:.1%}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 3.3 Classify Your Own Photo

Take a photo of something on your desk (a pen, a coffee mug, your phone, a pet) with your phone. Transfer it to your computer and upload it below.

MobileNetV2 knows 1,000 categories — see if it recognizes your object.

In [ ]:
from google.colab import files

print('Upload a photo from your phone or computer:')
uploaded = files.upload()

In [ ]:
filename = list(uploaded.keys())[0]
my_photo = Image.open(filename).convert('RGB')

plt.figure(figsize=(5, 5))
plt.imshow(my_photo)
plt.title('Your Photo')
plt.axis('off')
plt.show()

input_tensor = preprocess(my_photo).unsqueeze(0)

with torch.no_grad():
    output = model(input_tensor)

probabilities = torch.nn.functional.softmax(output[0], dim=0)
top5_prob, top5_idx = torch.topk(probabilities, 5)

print('\n=== Top 5 Predictions for Your Photo ===')
for i in range(5):
    class_name = categories[top5_idx[i].item()]
    confidence = top5_prob[i].item()
    print(f'{i+1}. {class_name:30s} {confidence:.1%}')

**Think about it:** MobileNetV2 was trained on everyday photos. What would happen if you fed it a chest X-ray? Try it — upload the CXR image you loaded in Part 2 and see what it predicts. This is why radiology AI models need to be **fine-tuned** on medical images.

---

## Part 4: Hand Landmark Detection, Segmentation, and Measurement

**Classification** answers: "What is in this image?"

**Detection** answers: "Where are the key points?" — it locates specific anatomical landmarks.

**Segmentation** answers: "Which pixels belong to what?" — it labels every pixel.

In radiology, these techniques are used to locate vertebrae, outline tumors, and measure organ dimensions. Here we use Google's **MediaPipe Hand Landmarker** to detect **21 landmark points** on each hand (fingertips, knuckles, wrist). From those landmarks we will:
1. Create a **segmentation mask** that isolates the hand from the background
2. **Measure each finger's length** — the same idea behind measuring tumor diameters or vertebral heights from AI-detected landmarks

### 4.1 Set Up the Hand Landmarker

In [ ]:
import mediapipe as mp
import cv2
import urllib.request
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# Download the hand landmarker model
urllib.request.urlretrieve(
    'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
    'hand_landmarker.task'
)

# Create the detector
base_options = mp_python.BaseOptions(model_asset_path='hand_landmarker.task')
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2
)
detector = mp_vision.HandLandmarker.create_from_options(options)

# Drawing utilities
mp_draw = mp.tasks.vision.drawing_utils
mp_styles = mp.tasks.vision.drawing_styles
mp_conns = mp.tasks.vision.HandLandmarksConnections

print('Hand Landmarker loaded.')
print('Landmark points per hand: 21')

In [ ]:
# ── Helper functions ──────────────────────────────────────────

def detect_hands(rgb_array):
    """Run hand landmark detection. Returns (annotated_image, result)."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_array)
    result = detector.detect(mp_image)
    annotated = np.copy(rgb_array)

    for idx, hand_lms in enumerate(result.hand_landmarks):
        mp_draw.draw_landmarks(
            annotated, hand_lms,
            mp_conns.HAND_CONNECTIONS,
            mp_styles.get_default_hand_landmarks_style(),
            mp_styles.get_default_hand_connections_style()
        )
        # Label left / right
        if idx < len(result.handedness):
            h, w = rgb_array.shape[:2]
            x = int(min(lm.x for lm in hand_lms) * w)
            y = int(min(lm.y for lm in hand_lms) * h) - 10
            label = result.handedness[idx][0].category_name
            cv2.putText(annotated, label, (x, max(y, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (88, 205, 54), 2)

    return annotated, result


def segment_hand(rgb_array, landmarks):
    """Create a convex-hull segmentation mask from hand landmarks."""
    h, w = rgb_array.shape[:2]
    points = np.array([(int(lm.x * w), int(lm.y * h)) for lm in landmarks])
    hull = cv2.convexHull(points)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(mask, hull, 255)
    segmented = rgb_array.copy()
    segmented[mask == 0] = [240, 240, 240]
    return mask, segmented


FINGER_JOINTS = {
    'Thumb':  [1, 2, 3, 4],
    'Index':  [5, 6, 7, 8],
    'Middle': [9, 10, 11, 12],
    'Ring':   [13, 14, 15, 16],
    'Pinky':  [17, 18, 19, 20],
}


def measure_fingers(landmarks, img_width, img_height):
    """Measure each finger's length in pixels from joint-to-joint distances."""
    lengths = {}
    for name, indices in FINGER_JOINTS.items():
        total = 0.0
        for i in range(len(indices) - 1):
            a = landmarks[indices[i]]
            b = landmarks[indices[i + 1]]
            dx = (a.x - b.x) * img_width
            dy = (a.y - b.y) * img_height
            total += math.sqrt(dx**2 + dy**2)
        lengths[name] = total
    return lengths


print('Helper functions defined: detect_hands, segment_hand, measure_fingers')

### 4.2 Detect Hand Landmarks

Upload a photo of your hand (palm facing camera, fingers spread) against a plain background.

In [ ]:
print('Upload a photo of your hand:')
hand_upload = files.upload()

In [ ]:
hand_filename = list(hand_upload.keys())[0]
hand_img = cv2.imread(hand_filename)
hand_rgb = cv2.cvtColor(hand_img, cv2.COLOR_BGR2RGB)

annotated, result = detect_hands(hand_rgb)

if result.hand_landmarks:
    n_hands = len(result.hand_landmarks)
    handedness = [result.handedness[i][0].category_name for i in range(n_hands)]
    print(f'Detected {n_hands} hand(s): {", ".join(handedness)}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(hand_rgb)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(annotated)
    axes[1].set_title('21 Hand Landmarks')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No hands detected. Try a clearer photo with your hand against a plain background.')

### 4.3 Create a Segmentation Mask

We use the 21 landmark points to create a **convex hull** — a polygon that wraps around all the points. Every pixel inside the hull is labeled as "hand" and every pixel outside is "background."

This is a simplified version of what clinical segmentation models do for tumors, organs, and other structures.

In [ ]:
if result.hand_landmarks:
    mask, segmented = segment_hand(hand_rgb, result.hand_landmarks[0])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(hand_rgb)
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Segmentation Mask')
    axes[1].axis('off')

    axes[2].imshow(segmented)
    axes[2].set_title('Segmented Hand')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    h, w = hand_rgb.shape[:2]
    hand_pixels = np.sum(mask > 0)
    print(f'Hand area: {hand_pixels:,} pixels ({hand_pixels / (h * w):.1%} of image)')
else:
    print('No hand was detected above. Re-upload a clearer photo.')

### 4.4 Measure Finger Lengths

AI landmark detection is not just for visualization — it lets you extract **quantitative measurements**. Here we measure each finger's length by summing the joint-to-joint distances along its landmarks.

This is the same principle behind measuring tumor diameters from segmentation contours, or vertebral body heights from spinal landmarks. The model gives you coordinates; the clinical measurement is computed from those coordinates.

In [ ]:
if result.hand_landmarks:
    h, w = hand_rgb.shape[:2]
    lengths = measure_fingers(result.hand_landmarks[0], w, h)

    # Absolute lengths in pixels
    print('=== Finger Lengths (pixels) ===')
    for name, px in lengths.items():
        print(f'  {name:8s} {px:6.1f} px')

    # Relative lengths (normalized so middle finger = 1.0)
    middle_len = lengths['Middle']
    print(f'\n=== Relative Lengths (Middle = 1.00) ===')
    for name, px in lengths.items():
        print(f'  {name:8s} {px / middle_len:.2f}')

    # Bar chart
    finger_names = list(lengths.keys())
    finger_px = list(lengths.values())
    finger_rel = [px / middle_len for px in finger_px]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    bar_colors = ['#F59E0B', '#0EA5E9', '#34D399', '#7C3AED', '#F43F5E']
    axes[0].bar(finger_names, finger_px, color=bar_colors)
    axes[0].set_ylabel('Length (pixels)')
    axes[0].set_title('Finger Lengths (absolute)')
    for i, v in enumerate(finger_px):
        axes[0].text(i, v + 2, f'{v:.0f}', ha='center', fontsize=10, fontweight='bold')

    axes[1].bar(finger_names, finger_rel, color=bar_colors)
    axes[1].set_ylabel('Relative to middle finger')
    axes[1].set_title('Finger Lengths (relative)')
    axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    for i, v in enumerate(finger_rel):
        axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
else:
    print('No hand detected.')

---

## Your Turn

Complete **all three tasks** below.

### Task 1: Try a Different Classifier

Train a **Random Forest** classifier on the radiology data. Random Forest builds many decision trees and combines their votes.

Use `RandomForestClassifier` from `sklearn.ensemble`. Compare its accuracy to the logistic regression and decision tree from Part 1.

In [ ]:
# ============================================================
# TASK 1: Train a Random Forest and compare accuracy
# Hint: from sklearn.ensemble import RandomForestClassifier
# ============================================================

from sklearn.ensemble import RandomForestClassifier

# Your code here:
# 1. Create the model: rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
# 2. Train it: rf_model.fit(X_train, y_train)
# 3. Predict: rf_predictions = rf_model.predict(X_test)
# 4. Print accuracy: accuracy_score(y_test, rf_predictions)



### Task 2: Classify Three Objects

Take photos of **three different objects** (e.g., a water bottle, a shoe, a houseplant). Upload and classify each one with MobileNetV2. For each, record:
- What the object actually is
- What MobileNetV2 predicted (top-1)
- The confidence score

In [ ]:
# ============================================================
# TASK 2: Upload and classify 3 different photos
# ============================================================

print('Upload 3 photos:')
task2_upload = files.upload()

for fname in task2_upload:
    img = Image.open(fname).convert('RGB')
    inp = preprocess(img).unsqueeze(0)

    with torch.no_grad():
        out = model(inp)

    probs = torch.nn.functional.softmax(out[0], dim=0)
    top_prob, top_idx = torch.topk(probs, 3)

    print(f'\n--- {fname} ---')
    for i in range(3):
        print(f'  {i+1}. {categories[top_idx[i].item()]:30s} {top_prob[i].item():.1%}')

### Task 3: Compare Two Hands

Take a photo of each hand separately (or both in one frame). For each hand:
1. Show the landmarks
2. Create the segmentation mask
3. Measure all five finger lengths
4. Create a side-by-side bar chart of finger lengths

Are your hands symmetric? Which finger differs the most between hands?

In [ ]:
# ============================================================
# TASK 3: Compare finger lengths between two hands
# ============================================================

print('Upload two hand photos (left hand and right hand):')
task3_upload = files.upload()

task3_files = list(task3_upload.keys())
all_lengths = []
hand_labels = []

fig, axes = plt.subplots(len(task3_files), 3, figsize=(15, 5 * len(task3_files)))
if len(task3_files) == 1:
    axes = axes[np.newaxis, :]

for row, fname in enumerate(task3_files):
    img = cv2.imread(fname)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ann, res = detect_hands(rgb)

    axes[row][0].imshow(ann)
    axes[row][0].set_title(f'{fname}', fontsize=10)
    axes[row][0].axis('off')

    if res.hand_landmarks:
        msk, seg = segment_hand(rgb, res.hand_landmarks[0])
        h, w = rgb.shape[:2]
        area_pct = np.sum(msk > 0) / (h * w)
        lengths = measure_fingers(res.hand_landmarks[0], w, h)
        all_lengths.append(lengths)

        label = res.handedness[0][0].category_name if res.handedness else f'Hand {row+1}'
        hand_labels.append(label)

        axes[row][1].imshow(msk, cmap='gray')
        axes[row][1].set_title(f'Mask ({area_pct:.1%})')
        axes[row][1].axis('off')

        axes[row][2].imshow(seg)
        axes[row][2].set_title(f'{label} hand')
        axes[row][2].axis('off')
    else:
        axes[row][1].text(0.5, 0.5, 'No hand detected', ha='center', va='center',
                          transform=axes[row][1].transAxes)
        axes[row][1].axis('off')
        axes[row][2].axis('off')

plt.tight_layout()
plt.show()

# Side-by-side finger length comparison
if len(all_lengths) == 2:
    finger_names = list(FINGER_JOINTS.keys())
    x = np.arange(len(finger_names))
    bar_w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - bar_w/2, [all_lengths[0][f] for f in finger_names], bar_w,
           label=hand_labels[0], color='#0EA5E9')
    ax.bar(x + bar_w/2, [all_lengths[1][f] for f in finger_names], bar_w,
           label=hand_labels[1], color='#F59E0B')
    ax.set_xticks(x)
    ax.set_xticklabels(finger_names)
    ax.set_ylabel('Length (pixels)')
    ax.set_title('Finger Length Comparison')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print('\n=== Difference (pixels) ===')
    for f in finger_names:
        diff = abs(all_lengths[0][f] - all_lengths[1][f])
        print(f'  {f:8s} {diff:6.1f} px')
elif len(all_lengths) == 1:
    print('Only one hand detected. Upload a second photo to compare.')

---

## Save Your Work

Run the completion record cell below, then save your notebook to GitHub.

### How to save:
1. In Colab, go to **File > Download > Download .ipynb**
2. Go to your forked repository: `https://github.com/YOUR-USERNAME/Bootcamp-AI-for-Medical-Imaging`
3. Click the **Code** tab, then **Add file > Upload files**
4. Drag and drop your downloaded `.ipynb` file
5. Type the commit message: `Completed Research Module 3`
6. Make sure **"Commit directly to the main branch"** is selected
7. Click **Commit changes**

In [ ]:
from datetime import datetime

print('=' * 50)
print('RESEARCH MODULE 3 \u2014 COMPLETION RECORD')
print('=' * 50)
print(f'GitHub Username:        {GITHUB_USERNAME}')
print(f'Completed:              {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Logistic Reg Accuracy:  {lr_accuracy:.1%}')
print(f'Decision Tree Accuracy: {dt_accuracy:.1%}')
print(f'MobileNetV2 classes:    {len(categories)}')
print(f'MediaPipe landmarks:    21')
print('=' * 50)
print('Save this notebook to GitHub to submit your work.')